# 🍳 Recipe 01 — LLM Basics & Prompt Engineering

> **AI Cookbook** by [Poorvi Bajpai](https://github.com/poorvibajpai)

---

## 🎯 What you'll learn
- What LLMs are and how they work (intuition)
- Zero-shot prompting
- Few-shot prompting
- Chain-of-Thought (CoT) reasoning
- Prompt templates using LangChain
- Model comparison: Groq (LLaMA 3) vs Gemini

**APIs used:** Groq, Google Gemini  
**Libraries:** `langchain`, `langchain-groq`, `langchain-google-genai`, `python-dotenv`

---

## 🧠 What is an LLM?

A **Large Language Model** is trained on massive text data to predict the next token.

```
Input:  'The capital of France is'
Output: 'Paris'  ← statistically most likely next token
```

Architecture:
```
Input Text → Tokenizer → Embeddings → Attention Layers → Output Tokens
```

Key insight: LLMs don't *know* things like a database. They generate statistically likely continuations based on training.

---

## ⚙️ Setup

> Make sure your `.env` file in the root folder has:
> ```
> GROQ_API_KEY=your_groq_key
> GOOGLE_API_KEY=your_gemini_key
> ```

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env from root folder

print('✅ API keys loaded!')
print('Groq key set:', bool(os.getenv('GROQ_API_KEY')))
print('Gemini key set:', bool(os.getenv('GOOGLE_API_KEY')))

✅ API keys loaded!
Groq key set: True
Gemini key set: True


In [4]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

groq_model = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)
gemini_model = ChatGoogleGenerativeAI(model='gemini-1.5-flash', temperature=0.7)

print('✅ Models ready!')
print('Groq   → llama3-8b-8192')
print('Gemini → gemini-1.5-flash')

✅ Models ready!
Groq   → llama3-8b-8192
Gemini → gemini-1.5-flash


---
## 📌 Part 1 — Zero-Shot Prompting

Ask the model with **no examples** — relies purely on its training data.

> Best for: simple, well-known tasks like translation, classification, summarization.

In [6]:
from langchain_core.messages import HumanMessage

zero_shot = "Classify the sentiment as Positive, Negative, or Neutral:\n\nReview: 'The food was amazing but the service was really slow.'"

print('PROMPT:', zero_shot)
print('\n' + '='*50)
print('\n🤖 Groq:', 
groq_model.invoke([HumanMessage(content=zero_shot)]).content)
gemini_model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

PROMPT: Classify the sentiment as Positive, Negative, or Neutral:

Review: 'The food was amazing but the service was really slow.'


🤖 Groq: Neutral. 

The review contains both positive and negative comments. The phrase "the food was amazing" is positive, while "the service was really slow" is negative. Since both opinions are present and balance each other out, the overall sentiment is neutral.


### 🔍 Observation
Both models classify correctly with zero guidance. Works because sentiment analysis is heavily represented in training data.

Notice: responses may differ in format and reasoning — that's normal with different models.

---
## 📌 Part 2 — Few-Shot Prompting

Give **2–5 examples** before your question to teach the model your exact format/style.

> Best for: custom output formats, consistent structure, domain-specific tasks.

In [8]:
few_shot = """Classify sentiment as Positive 😊, Negative 😞, or Neutral 😐.
Format: Sentiment: <label> | Reason: <one sentence>

Examples:
Review: 'Best pizza I have ever had!' → Sentiment: Positive 😊 | Reason: Strong praise with superlative.
Review: 'The package arrived damaged.' → Sentiment: Negative 😞 | Reason: Describes a bad experience.
Review: 'The product is okay, nothing special.' → Sentiment: Neutral 😐 | Reason: Neither strongly positive nor negative.

Now classify:
Review: 'The laptop is fast but the battery drains too quickly.'"""

print('🤖 Groq:', groq_model.invoke([HumanMessage(content=few_shot)]).content)
# print('\n✨ Gemini:', gemini_model.invoke([HumanMessage(content=few_shot)]).content)

🤖 Groq: Sentiment: Neutral 😐 | Reason: The review mentions both a positive aspect ('fast') and a negative aspect ('battery drains too quickly'), balancing the sentiment.


### 🔍 Observation
Few-shot forces the model to follow your exact output format. Compare with Part 1 — zero-shot was free-form, few-shot is structured and consistent.

This is crucial in production when you need **parseable, predictable output**.

---
## 📌 Part 3 — Chain-of-Thought (CoT) Prompting

Adding **"Think step by step"** makes the model show its reasoning before answering.

> Best for: math, logic puzzles, multi-step reasoning.

In [9]:
problem = """A store sells apples for Rs.5 each and oranges for Rs.8 each.
Priya buys 4 apples and 3 oranges. She pays with a Rs.100 note.
How much change does she get back?"""

print('❌ WITHOUT CoT:')
print('-'*40)
print(groq_model.invoke([HumanMessage(content=problem)]).content)

print('\n✅ WITH CoT (Think step by step):')
print('-'*40)
print(groq_model.invoke([HumanMessage(content=problem + '\n\nThink step by step.')]).content)

❌ WITHOUT CoT:
----------------------------------------
To find out how much change Priya gets back, we need to first calculate the total cost of the items she bought. 

The cost of 4 apples is 4 * Rs.5 = Rs.20.
The cost of 3 oranges is 3 * Rs.8 = Rs.24.
Total cost = Rs.20 + Rs.24 = Rs.44.

Since Priya paid with a Rs.100 note, her change will be Rs.100 - Rs.44 = Rs.56.

So, Priya gets Rs.56 as change.

✅ WITH CoT (Think step by step):
----------------------------------------
To find out how much change Priya gets back, we need to calculate the total cost of the items she bought and then subtract that from the amount she paid with.

Step 1: Calculate the cost of apples
- The cost of 1 apple is Rs.5.
- Priya buys 4 apples, so the total cost of apples is 4 x Rs.5 = Rs.20.

Step 2: Calculate the cost of oranges
- The cost of 1 orange is Rs.8.
- Priya buys 3 oranges, so the total cost of oranges is 3 x Rs.8 = Rs.24.

Step 3: Calculate the total cost of the items
- Total cost = cost of apple

### 🔍 Observation
With CoT: `4 × Rs.5 = Rs.20`, `3 × Rs.8 = Rs.24`, `total = Rs.44`, `change = Rs.56`

CoT makes answers **auditable and more reliable** — especially on harder problems.

---
## 📌 Part 4 — Prompt Templates with LangChain

`ChatPromptTemplate` = reusable, parameterized prompts. Like f-strings, but for LLMs.

> Use this in production - never hardcode prompts as raw strings.

In [11]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ('system', 'You are an expert {domain} tutor. Explain clearly for a {level} student.'),
    ('human', 'Explain: {concept}')
])

# Build a chain: template → model (LCEL syntax)
chain = template | groq_model

inputs = [
    {'domain': 'Machine Learning', 'level': 'beginner', 'concept': 'What is overfitting?'},
    {'domain': 'Python', 'level': 'intermediate', 'concept': 'How do decorators work?'},
]

for inp in inputs:
    print(f"\n📚 {inp['concept']}")
    print('-'*40)
    print(chain.invoke(inp).content)


📚 What is overfitting?
----------------------------------------
Welcome to the world of Machine Learning! I'm excited to help you understand this crucial concept.

**What is Overfitting?**

Overfitting is a common problem in Machine Learning where a model is too complex and performs well on the training data, but poorly on new, unseen data. It's like trying to fit a curve that perfectly passes through every single point on a graph, but ends up being so wiggly that it fails to capture the underlying pattern.

**Why does Overfitting happen?**

Overfitting occurs when a model is too complex and tries to learn the noise (random variations) in the training data, rather than the underlying patterns. This is because complex models can fit the training data too well, but they don't generalize well to new data.

**Types of Overfitting:**

1. **Feature Overfitting:** When a model has too many features and tries to learn the relationships between all of them, it can lead to overfitting.
2. **Mod

---
## 📌 Part 5 — Model Comparison: Groq vs Gemini

Same prompt, two models. Observe differences in style, depth, and word count.

In [13]:
prompt = 'In 3 bullet points, explain why RAG is better than fine-tuning for most real-world use cases.'

gr = groq_model.invoke([HumanMessage(content=prompt)])
# gm = gemini_model.invoke([HumanMessage(content=prompt)])

print('🤖 Groq (LLaMA 3-8B):')
print('-'*40)
print(gr.content)

# print('\n✨ Gemini (1.5 Flash):')
# print('-'*40)
# print(gm.content)

print('\n📊 Word count:')
print(f'Groq:   {len(gr.content.split())} words')
# print(f'Gemini: {len(gm.content.split())} words')

🤖 Groq (LLaMA 3-8B):
----------------------------------------
I must note that the statement "RAG is better than fine-tuning for most real-world use cases" is a subjective comparison, and the choice between RAG (Reptile and Adapter-based Generalization) and fine-tuning ultimately depends on the specific use case and requirements. However, here are three points that highlight some advantages of RAG over fine-tuning:

* **Efficient knowledge transfer**: RAG is designed to efficiently transfer knowledge from a pre-trained model to a downstream task, without the need for extensive fine-tuning. This makes it particularly useful when working with large, complex models or when fine-tuning is computationally expensive. By leveraging the adapter mechanism, RAG can adapt the pre-trained model to the downstream task more quickly and efficiently than traditional fine-tuning methods.
* **Improved generalizability**: RAG's adapter-based approach allows it to generalize better to new tasks and domain

---
## 🏆 Mini Challenge

Try these yourself before moving to Recipe 02:

1. **Zero-shot**: Ask the model to translate a sentence to Hindi
2. **Few-shot**: Create a prompt that outputs structured JSON
3. **CoT**: Give it a logic puzzle — compare with/without `'Think step by step'`
4. **Template**: Build a prompt template for a **code review assistant**

---

## 📝 Key Takeaways

| Technique | When to use | Magic phrase |
|-----------|-------------|-------------|
| Zero-shot | Simple, common tasks | Just ask directly |
| Few-shot | Custom format / style | Show 2–5 examples |
| Chain-of-Thought | Math, logic, reasoning | "Think step by step" |
| Prompt Templates | Reusable production prompts | `ChatPromptTemplate` |

---

**Next Recipe →** `02-sentiment-pipeline/notebook.ipynb`